# FastSpeech2 fine-tuning runbook (Colab / Kaggle GPU)
Run the cells **top to bottom**. Set `LANG` once in cell 2 - run the whole notebook once for `English`, then again for `Telugu`.
Runtime -> Change runtime type -> **T4 GPU**. Budget: preprocessing ~15-30 min, training ~1.5-3 h per language.

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout or 'NO GPU - change runtime type!')
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
LANG = "English"          # <-- "English" or "Telugu"
REPO_URL = "https://github.com/<YOUR-USER>/<YOUR-REPO>.git"   # <-- your repo
HF_TOKEN = ""             # <-- Hugging Face token (needed for Svarah: accept its terms on the dataset page first)
USE_DRIVE = True          # keep checkpoints on Google Drive so a disconnect does not lose the run

In [ ]:
import os
if USE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    CKPT_DIR = f'/content/drive/MyDrive/fs2_{LANG}_ckpt'
else:
    CKPT_DIR = f'/content/fs2_{LANG}_ckpt'
os.makedirs(CKPT_DIR, exist_ok=True)
!rm -rf /content/project && git clone {REPO_URL} /content/project
%cd /content/project
!pip install -q -r requirements.txt
!apt-get -qq install -y fonts-noto-core > /dev/null
os.environ['HF_TOKEN'] = HF_TOKEN
%cd /content/project/{LANG}

## 1. Pre-processing (alignment + mel / pitch / energy / duration)
Sanity-check first with 40 utterances (1-2 min). Most outcomes should be `ok`.

In [ ]:
!python preprocess.py -c config.yaml --max_utts 40 --restart

In [ ]:
!python preprocess.py -c config.yaml --restart      # full run; resumable (drop --restart to resume)

In [ ]:
!cat preprocessed_data/*/preprocess_summary.json

## 2. Optional: pretrained LJSpeech FastSpeech2 for warm start
Download `900000.pth.tar` from the upstream pretrained-models Drive folder
(https://drive.google.com/drive/folders/1DOhZGlTLMbbAAFZmZGDdc77kz1PloS7F) into your Drive and set the path below.
Without it (`PRETRAINED = None`) training from scratch also works but needs more steps.

In [ ]:
PRETRAINED = "/content/drive/MyDrive/LJSpeech_900000.pth.tar"   # or None
if PRETRAINED and not os.path.exists(PRETRAINED): print("not found -> training from scratch"); PRETRAINED = None

## 3. Train
If the runtime dies: re-run the setup cells and preprocessing (minutes), then use the resume cell.

In [ ]:
pre = f"--pretrained {PRETRAINED}" if PRETRAINED else ""
!python train.py -c config.yaml --ckpt_path {CKPT_DIR} {pre}

In [ ]:
# RESUME after a disconnect: set N to the newest step file in CKPT_DIR (e.g. 7500)
N = 0
if N: get_ipython().system(f"python train.py -c config.yaml --ckpt_path {CKPT_DIR} --restore_step {N}")

In [ ]:
!python train.py -c config.yaml --plot_only      # re-draw loss plots from the CSV logs

## 4. Inference on the 5 test sentences (audio + duration / pitch / energy plots)

In [ ]:
!python inference.py -c config.yaml

In [ ]:
from IPython.display import Audio, display
import glob
for f in sorted(glob.glob(f'../Results/{LANG}/test_generated_audios/*.wav')):
    print(f); display(Audio(f))

## 5. Zip results, download, copy into your repo, commit

In [ ]:
!cd /content/project && zip -qr /content/{LANG}_outputs.zip {LANG}/checkpoints/*.pth.tar {LANG}/checkpoints/logs/*.csv {LANG}/checkpoints/logs/*.json Results/{LANG} {LANG}/preprocessed_data/*/preprocess_summary.json
from google.colab import files; files.download(f'/content/{LANG}_outputs.zip')